# 🎧 NexTune — Bluetooth Headphones Price Prediction
## Scrape → EDA → Clean → ML → Predict

**Problem:** Your company is launching a new wireless Bluetooth headphone in the Indian market. Recommend a suitable price based on the product's specifications and market demand from Amazon India.

---
> **▶️ HOW TO RUN:** Open in [Google Colab](https://colab.research.google.com) → Runtime → Run all

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn joblib requests

In [ ]:
import re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
print("✅ Libraries loaded")

---
## 🕷️ STEP 1 — Scrape: Load Data from GitHub

We scraped Amazon India using BeautifulSoup + Selenium with stealth mode (`undetected-chromedriver`). The scraper used regex patterns to extract structured features from unstructured product titles — this is the **prompt engineering / enhanced scraping** step.

Below we show the regex patterns used, then load the cleaned dataset directly from GitHub.

In [ ]:
# Regex patterns used by our enhanced scraper (src/scrapers/enhanced_scraper.py)
SCRAPER_PATTERNS = {
    "battery_life"     : r"(\d+)\s*(?:hours?|hrs?|h)\s*(?:battery|playtime|playback)?",
    "bluetooth_version": r"bluetooth\s*(?:v|version)?\s*([0-9]\.[0-9])",
    "driver_size"      : r"(\d+)\s*mm\s*driver",
    "anc_db"           : r"(\d+)\s*db\s*(?:anc|noise)",
    "mic_count"        : r"(\d+)\s*mic",
    "ipx_rating"       : r"ipx(\d+)",
}

# Demo on a real product title
sample = "boAt Rockerz 551 ANC Pro Bluetooth v5.4 Over Ear 80H Battery 40dB ANC 4Mic IPX4"
print(f"Sample title:\n  {sample}\n")
print("Extracted features:")
for field, pattern in SCRAPER_PATTERNS.items():
    m = re.search(pattern, sample, re.IGNORECASE)
    print(f"  {field:20s}: {m.group(1) if m else 'not found'}")

In [ ]:
import os

# Auto-detect environment: Colab or local
if os.path.exists('/content'):
    # Running in Google Colab — load from GitHub
    DATA_PATH = "https://raw.githubusercontent.com/ESpoorthy/NexTune/main/data/enhanced-headphones-dataset.csv"
else:
    # Running locally in VS Code — use relative path
    DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), 'data', 'enhanced-headphones-dataset.csv')
    if not os.path.exists(DATA_PATH):
        # fallback: try from notebook's own directory
        DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('.')), 'data', 'enhanced-headphones-dataset.csv')

print(f"Loading from: {DATA_PATH}")
df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset shape  : {df_raw.shape}")
print(f"Columns        : {df_raw.columns.tolist()}")
df_raw.head(3)

---
## 📊 STEP 2 — EDA: Exploratory Data Analysis

Understand the raw data before touching it.

In [ ]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
df_raw.info()
print("\nNumerical Summary:")
df_raw.describe().round(2)

In [ ]:
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({"Missing": missing, "% Missing": missing_pct})
missing_df = missing_df[missing_df["Missing"] > 0].sort_values("% Missing", ascending=False)

plt.figure(figsize=(12, 4))
sns.barplot(x=missing_df["% Missing"], y=missing_df.index, palette="Reds_r")
plt.title("% Missing Values per Feature", fontsize=14, fontweight="bold")
plt.xlabel("% Missing")
plt.tight_layout()
plt.show()
print(missing_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_raw["price_inr"].dropna(), bins=50, color="steelblue", edgecolor="white")
axes[0].axvline(df_raw["price_inr"].median(), color="red", linestyle="--",
                label=f'Median ₹{df_raw["price_inr"].median():.0f}')
axes[0].set_title("Price Distribution (Raw)", fontweight="bold")
axes[0].set_xlabel("Price (INR)")
axes[0].legend()

axes[1].boxplot(df_raw["price_inr"].dropna())
axes[1].set_title("Price Box Plot", fontweight="bold")
axes[1].set_ylabel("Price (INR)")
plt.tight_layout()
plt.show()

print(f"Mean   : ₹{df_raw.price_inr.mean():.0f}")
print(f"Median : ₹{df_raw.price_inr.median():.0f}")
print(f"Min    : ₹{df_raw.price_inr.min():.0f}")
print(f"Max    : ₹{df_raw.price_inr.max():.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_counts = df_raw["category"].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values, color=["#FF6B6B","#4ECDC4","#45B7D1","#FFA07A"])
axes[0].set_title("Products per Category", fontweight="bold")
axes[0].tick_params(axis="x", rotation=30)

df_raw.boxplot(column="price_inr", by="category", ax=axes[1])
axes[1].set_title("Price by Category", fontweight="bold")
axes[1].set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

top_brands = df_raw["brand"].value_counts().head(15)
plt.figure(figsize=(12, 5))
sns.barplot(x=top_brands.values, y=top_brands.index, palette="coolwarm")
plt.title("Top 15 Brands by Product Count", fontweight="bold")
plt.xlabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df_raw["battery_life_hrs"], df_raw["price_inr"], alpha=0.5, color="teal")
axes[0].set_title("Price vs Battery Life", fontweight="bold")
axes[0].set_xlabel("Battery Life (hrs)")
axes[0].set_ylabel("Price (INR)")

axes[1].scatter(df_raw["rating"], df_raw["price_inr"], alpha=0.5, color="coral")
axes[1].set_title("Price vs Rating", fontweight="bold")
axes[1].set_xlabel("Rating")
axes[1].set_ylabel("Price (INR)")
plt.tight_layout()
plt.show()

---
## 🔍 STEP 3 — Understand: Correlation & Key Insights

In [ ]:
num_cols = ["price_inr","rating","review_count","battery_life_hrs","driver_size_mm","bluetooth_version","mic_count"]
corr = df_raw[num_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Feature Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nCorrelation with Price (sorted):")
print(corr["price_inr"].sort_values(ascending=False).to_string())

In [ ]:
print("=" * 60)
print("KEY INSIGHTS FROM EDA")
print("=" * 60)
print(f"Total products       : {len(df_raw)}")
print(f"Price range          : ₹{df_raw.price_inr.min():.0f} – ₹{df_raw.price_inr.max():.0f}")
print(f"Median price         : ₹{df_raw.price_inr.median():.0f}")
print(f"Unique brands        : {df_raw.brand.nunique()}")
anc_mask = df_raw.active_noise_cancellation == 1
print(f"ANC products         : {anc_mask.sum()} ({anc_mask.mean()*100:.1f}%)")
print(f"Avg ANC price        : ₹{df_raw[anc_mask].price_inr.mean():.0f}")
print(f"Avg non-ANC price    : ₹{df_raw[~anc_mask].price_inr.mean():.0f}")
print(f"Most common category : {df_raw.category.mode()[0]}")
print("=" * 60)

---
## 🧹 STEP 4 — Clean: Preprocessing & Feature Engineering

In [ ]:
df = df_raw.copy()
print(f"Before cleaning: {len(df)} rows")

# Drop rows missing price or brand
df = df.dropna(subset=["price_inr", "brand"])
df = df[df["price_inr"] > 0]

# Remove duplicates
df = df.drop_duplicates(subset=["product_name"], keep="first")
print(f"After removing nulls + duplicates: {len(df)} rows")

# Impute missing values
df["battery_life_hrs"] = df.groupby("category")["battery_life_hrs"].transform(lambda x: x.fillna(x.median()))
df["bluetooth_version"] = df["bluetooth_version"].fillna(df["bluetooth_version"].mode()[0])
df["driver_size_mm"]    = df.groupby("category")["driver_size_mm"].transform(lambda x: x.fillna(x.median()))
df["rating"]            = df["rating"].fillna(df["rating"].median())
df["review_count"]      = df["review_count"].fillna(0)
df["active_noise_cancellation"] = df["active_noise_cancellation"].fillna(0)
df["mic_count"]         = df["mic_count"].fillna(2)

print("\nMissing values in key columns after cleaning:")
for c in ["price_inr","rating","battery_life_hrs","bluetooth_version","driver_size_mm"]:
    print(f"  {c}: {df[c].isnull().sum()}")

In [ ]:
# New features
df["has_anc"]        = (df["active_noise_cancellation"] == 1).astype(int)
df["price_per_hour"] = df["price_inr"] / (df["battery_life_hrs"] + 1)
df["high_rating"]    = (df["rating"] >= 4.0).astype(int)
df["has_ipx"]        = df["ipx_rating"].notna().astype(int)
df["bt_major"]       = df["bluetooth_version"].apply(lambda x: int(x) if pd.notna(x) else 5)

# Brand tier
brand_avg = df.groupby("brand")["price_inr"].mean()
def brand_tier(b):
    p = brand_avg.get(b, 0)
    return "premium" if p >= 10000 else ("mid" if p >= 3000 else "budget")
df["brand_tier"] = df["brand"].apply(brand_tier)

print("✅ Feature engineering done")
print(f"  has_anc    : {df.has_anc.sum()} products with ANC")
print(f"  high_rating: {df.high_rating.sum()} products rated ≥ 4.0")
print(f"  has_ipx    : {df.has_ipx.sum()} products with IPX rating")
print(f"  brand_tier : {df.brand_tier.value_counts().to_dict()}")

In [ ]:
le_dict = {}
for col in ["category", "brand", "brand_tier"]:
    le = LabelEncoder()
    df[f"{col}_enc"] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

FEATURES = [
    "category_enc", "brand_enc", "brand_tier_enc",
    "rating", "review_count",
    "battery_life_hrs", "driver_size_mm",
    "bluetooth_version", "bt_major",
    "mic_count", "has_anc", "has_ipx",
    "high_rating", "price_per_hour"
]
TARGET = "price_inr"

X = df[FEATURES].fillna(0)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Feature matrix : {X.shape}")
print(f"Train / Test   : {len(X_train)} / {len(X_test)}")

---
## 🤖 STEP 5 — ML: Train, Evaluate & Select Best Model

In [ ]:
models = {
    "Linear Regression" : LinearRegression(),
    "Ridge"             : Ridge(alpha=1.0),
    "Lasso"             : Lasso(alpha=1.0),
    "Random Forest"     : RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    "Gradient Boosting" : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1))) * 100
    cv   = cross_val_score(model, X_train_sc, y_train, cv=5, scoring="r2").mean()
    results.append({"Model":name, "R²":r2, "CV R²":cv, "RMSE":rmse, "MAE":mae, "MAPE%":mape})
    print(f"{name:22s}  R²={r2:.3f}  CV R²={cv:.3f}  RMSE=₹{rmse:.0f}  MAE=₹{mae:.0f}")

results_df = pd.DataFrame(results).sort_values("R²", ascending=False).reset_index(drop=True)
print("\n--- Ranked Results ---")
print(results_df.round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ["#2ecc71" if r >= 0.70 else "#e74c3c" for r in results_df["R²"]]
axes[0].barh(results_df["Model"], results_df["R²"], color=colors)
axes[0].axvline(0.70, color="black", linestyle="--", label="Target R²=0.70")
axes[0].set_title("R² Score", fontweight="bold")
axes[0].legend()

axes[1].barh(results_df["Model"], results_df["RMSE"], color="steelblue")
axes[1].set_title("RMSE (₹) — lower is better", fontweight="bold")

axes[2].barh(results_df["Model"], results_df["MAPE%"], color="coral")
axes[2].set_title("MAPE% — lower is better", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
best_name  = results_df.iloc[0]["Model"]
best_model = models[best_name]
y_pred_best = best_model.predict(X_test_sc)

print(f"🏆 Best Model : {best_name}")
print(f"   R²         : {results_df.iloc[0]['R²']:.4f}")
print(f"   RMSE       : ₹{results_df.iloc[0]['RMSE']:.2f}")
print(f"   MAE        : ₹{results_df.iloc[0]['MAE']:.2f}")

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.6, color="teal", edgecolors="white", s=70)
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
plt.plot(lims, lims, "r--", lw=2, label="Perfect prediction")
plt.xlabel("Actual Price (₹)", fontsize=12)
plt.ylabel("Predicted Price (₹)", fontsize=12)
plt.title(f"{best_name} — Actual vs Predicted", fontsize=14, fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
if hasattr(best_model, "feature_importances_"):
    imp = best_model.feature_importances_
    label = "Feature Importance"
else:
    imp = np.abs(best_model.coef_)
    label = "|Coefficient|"

feat_imp = pd.Series(imp, index=FEATURES).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
feat_imp.plot(kind="barh", color="#378ADD")
plt.title(f"What Factors Affect Price Most? ({best_name})", fontsize=14, fontweight="bold")
plt.xlabel(label)
plt.tight_layout()
plt.show()

print("\nTop 5 most important features:")
print(feat_imp.sort_values(ascending=False).head(5).round(4).to_string())

---
## 💾 STEP 6 — Save Model

In [ ]:
import os
from google.colab import files

os.makedirs("/content/models", exist_ok=True)
joblib.dump(best_model, "/content/models/price_predictor.pkl")
joblib.dump(scaler,     "/content/models/scaler.pkl")
joblib.dump(le_dict,    "/content/models/label_encoders.pkl")

print("✅ Saved: price_predictor.pkl")
print("✅ Saved: scaler.pkl")
print("✅ Saved: label_encoders.pkl")

# Download to your machine
files.download("/content/models/price_predictor.pkl")
files.download("/content/models/scaler.pkl")
files.download("/content/models/label_encoders.pkl")

---
## 💡 STEP 7 — Predict Price for a New Headphone

Fill in your new product specs and run to get a recommended market price.

In [ ]:
def predict_price(brand, has_anc, is_tws, has_mic,
                  is_neckband, is_over_ear, has_bass,
                  battery_hrs=30, driver_mm=10, bt_version=5.3,
                  rating=4.0, reviews=0):
    # Determine category from flags
    if is_tws:
        category = "true wireless earbuds"
    elif is_neckband:
        category = "neckband"
    else:
        category = "over-ear headphone"

    # Determine brand tier
    tier = brand_tier(brand)

    # Encode
    cat_enc   = le_dict["category"].transform([category])[0] if category in le_dict["category"].classes_ else 0
    brand_enc = le_dict["brand"].transform([brand])[0] if brand in le_dict["brand"].classes_ else 0
    tier_enc  = le_dict["brand_tier"].transform([tier])[0]

    price_per_hour = 0  # unknown for new product

    input_vec = np.array([[
        cat_enc, brand_enc, tier_enc,
        rating, reviews,
        battery_hrs, driver_mm,
        bt_version, int(bt_version),
        2 if has_mic else 0,
        int(has_anc), 0,
        int(rating >= 4.0), price_per_hour
    ]])

    input_scaled = scaler.transform(input_vec)
    price = best_model.predict(input_scaled)[0]

    print("\n" + "=" * 45)
    print("       PRICE RECOMMENDATION")
    print("=" * 45)
    print(f"  Brand        : {brand}")
    print(f"  Category     : {category}")
    print(f"  Brand Tier   : {tier}")
    print(f"  ANC          : {'Yes' if has_anc else 'No'}")
    print(f"  TWS          : {'Yes' if is_tws else 'No'}")
    print(f"  Mic          : {'Yes' if has_mic else 'No'}")
    print(f"  Battery      : {battery_hrs}h")
    print(f"  Driver       : {driver_mm}mm")
    print(f"  Bluetooth    : v{bt_version}")
    print("-" * 45)
    print(f"  Predicted    : ₹{price:,.0f}")
    print(f"  Suggested    : ₹{price*0.9:,.0f} – ₹{price*1.1:,.0f}  (±10% band)")
    print("=" * 45)

# ✏️ Edit these specs for your new headphone
predict_price(
    brand="Boat",
    has_anc=True,
    is_tws=True,
    has_mic=True,
    is_neckband=False,
    is_over_ear=False,
    has_bass=True,
    battery_hrs=40,
    driver_mm=10,
    bt_version=5.3,
    rating=4.2,
    reviews=0
)